### Building a RAG System with LangChain and ChromaDB

## Introduction

Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- **LangChain:** A framework for developing applications powered by language models  
- **ChromaDB:** An open-source vector database for storing and retrieving embeddings  
- **OpenAI:** For embeddings and language model (you can substitute with other providers) I use Groq api key
- **Hugging Face:** Free and open-source models for embeddings and inference

In [1]:
import os
from dotenv import load_dotenv

# Load .env first
load_dotenv()

# Check key
print(os.getenv("GROQ_API_KEY"))

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


In [2]:
# =========================
# LangChain Imports
# =========================

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# =========================
# Document Loaders
# =========================

from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader
)

# =========================
# Embedding Models
# =========================

from langchain_huggingface import HuggingFaceEmbeddings

# Optional OpenAI embeddings
# from langchain_openai import OpenAIEmbeddings

# =========================
# Vector Store
# =========================

from langchain_community.vectorstores import Chroma

# =========================
# Utilities
# =========================

import os
import numpy as np
from typing import List

# =========================
# Hugging Face Embedding Model
# =========================

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("All imports successful 🚀")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

All imports successful 🚀


In [3]:
# RAG Architecture Overview

print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



### 1. Sample Data

In [4]:
## create sample documents

sample_docs = [
    """
    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement
    learning. Supervised learning uses labeled data to train models, while unsupervised
    learning finds patterns in unlabeled data. Reinforcement learning learns through
    interaction with an environment using rewards and penalties.
    """,

    """
    Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of interconnected
    nodes. Deep learning has revolutionized fields like computer vision, natural language
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers
    excel at sequential data processing.
    """,

    """
    Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language.
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis,
    machine translation, and question answering. Modern NLP heavily relies on transformer
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand
    context and relationships between words in text.
    """
]

sample_docs

['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning uses labeled data to train models, while unsupervised\n    learning finds patterns in unlabeled data. Reinforcement learning learns through\n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly\n    effective for image 

In [5]:
# save sample documents to text files
import tempfile
temp_dir =tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt", "w") as f:
        f.write(doc)
        
print(f"Sample documents create in: {temp_dir}")

Sample documents create in: C:\Users\ABHINE~1\AppData\Local\Temp\tmpk6zn24qo


In [6]:
temp_dir

'C:\\Users\\ABHINE~1\\AppData\\Local\\Temp\\tmpk6zn24qo'

In [7]:
# save sample documents to text files
import tempfile
temp_dir =tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt", "w") as f:
        f.write(doc)
        

### 2. Document Loading


In [8]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# load documents from the directory
loader =DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents =loader.load()

print(f"Loaded {len(documents)} documents:")
print(f"\nFirst Document Preview")
print(documents[0].page_content[:200] + "...")
 

Loaded 3 documents:

First Document Preview

    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There...


In [9]:
documents

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning uses labeled data to train models, while unsupervised\n    learning finds patterns in unlabeled data. Reinforcement learning learns through\n    interaction with an environment using rewards and penalties.\n    '),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural lang

### 3. Document Splitting

In [10]:
# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap =50,
    length_function = len,
    separators = [" "]
)

chunks=text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks from {len(documents)} documents.")
print(f"\nFirst Chunk Example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")


Created 5 chunks from 3 documents.

First Chunk Example:
Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experien...
Metadata: {'source': 'data\\doc_0.txt'}


In [11]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning uses labeled data to train models, while unsupervised\n    learning finds patterns in unlabeled data. Reinforcement learning learns through\n    interaction'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='learning learns through\n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnec

### 4. Embedding models

In [12]:
sample_text = "Machine Learning is fantastic."
embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"

)
embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [13]:
vector = embeddings.embed_query(sample_text)
vector

[-0.012726830318570137,
 -0.0818236917257309,
 0.05881974846124649,
 0.001749892602674663,
 -0.02922222390770912,
 -0.0053299409337341785,
 -0.09802888333797455,
 -0.05204655975103378,
 -0.0587361715734005,
 -0.0205682460218668,
 -0.02951933443546295,
 0.0678846463561058,
 0.060227639973163605,
 0.002353089628741145,
 -0.07194127142429352,
 0.013093036599457264,
 -0.02969030663371086,
 -0.03324619308114052,
 -0.029223233461380005,
 -0.14643634855747223,
 -0.006786899641156197,
 -0.02649769000709057,
 0.015934431925415993,
 -0.026386870071291924,
 0.02343549206852913,
 0.03558555245399475,
 0.021617334336042404,
 -0.0023313811980187893,
 -0.014821548946201801,
 -0.034824520349502563,
 -0.010180369019508362,
 0.04471692070364952,
 -0.013622415252029896,
 0.024015655741095543,
 -0.12836576998233795,
 0.01006049383431673,
 0.007377081550657749,
 0.052105385810136795,
 0.029393240809440613,
 0.023675134405493736,
 -0.01997978426516056,
 -0.020756741985678673,
 0.01653829962015152,
 -0.00035

### Initialize the ChromaDB Vector Store And Stores the chunks in Vector Representation

In [14]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning uses labeled data to train models, while unsupervised\n    learning finds patterns in unlabeled data. Reinforcement learning learns through\n    interaction'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='learning learns through\n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnec

In [15]:
## create a Chromadb vector store

persist_directory = "./chroma_db"

## Initialize Chromadb with HuggingFace embeddings

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(),
    persist_directory= persist_directory,
    collection_name="rag_collection"
)

print(f"Vector store create with {vectorstore._collection.count()} vectors")
print(f"Persisted to {persist_directory}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector store create with 34 vectors
Persisted to ./chroma_db


###

### Test Similarity Search 

In [16]:
query = "What are the types of Machine learning?"

similar_docs = vectorstore.similarity_search(query,k=3)
similar_docs



[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning uses labeled data to train models, while unsupervised\n    learning finds patterns in unlabeled data. Reinforcement learning learns through\n    interaction'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning uses labeled data to train

In [17]:
query = "What is NLP?"

similar_docs = vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language.\n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis,\n    machine translation, and question answering. Modern NLP heavily relies on transformer\n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand\n    context and relationships between words in text.'),
 Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language.\n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis,\n    machine translation, and question answering. Modern NLP heavily relies on transformer\n    architectures like BERT, GPT, and T5. These models use attention mechanisms

In [18]:
query = "What is Deep Learning?"

similar_docs = vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly\n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Netwo

In [19]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunks {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source','unknow')}")

Query: What is Deep Learning?

Top 3 similar chunks

--- Chunks 1 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of in...
Source: data\doc_1.txt

--- Chunks 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of in...
Source: data\doc_1.txt

--- Chunks 3 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of in...
Source: data\doc_1.txt


### Advanced Similarity Search  with Score

In [20]:
result_score = vectorstore.similarity_search_with_score(query,k=3)
result_score

[(Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly\n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
  0.49731138348579407),
 (Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. 

##  Understanding Similarity Scores

The similarity score represents how closely related a document chunk is to your query.  
The scoring depends on the distance metric used:

### ChromaDB Default
Uses **L2 distance (Euclidean distance)**

- Lower scores = MORE similar (closer in vector space)
- Score of `0` = identical vectors
- Typical range: `0 to 2` (but can be higher)

---

### Cosine Similarity (if configured)

- Higher scores = MORE similar
- Range: `-1 to 1`
- `1` means identical vectors

### Initialize LLM, RAG Chain Prompt  Template , Query the RAG system

In [21]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model_name= "llama-3.1-8b-instant",
    #temperature=0.2,
    #max_tokens=500
)

In [22]:
test_response = llm.invoke("What is LLM")
test_response

AIMessage(content='LLM stands for Large Language Model. It is a type of artificial intelligence (AI) model that is designed to process and generate human-like language. LLMs are trained on vast amounts of text data, which allows them to learn patterns, relationships, and structures of language.\n\nLarge Language Models are typically trained using a type of machine learning called deep learning, which involves multiple layers of artificial neural networks. These models are capable of learning complex patterns and relationships in language, such as:\n\n1. **Syntax**: The rules that govern how words are arranged to form sentences and phrases.\n2. **Semantics**: The meaning of words, phrases, and sentences.\n3. **Pragmatics**: The context and implications of language use.\n\nLLMs can be used for a wide range of applications, including:\n\n1. **Language translation**: LLMs can translate text from one language to another.\n2. **Text summarization**: LLMs can summarize long pieces of text int

In [23]:
from langchain.chat_models import init_chat_model # LangChain me universal chat model initialize karne ke liye use hota hai.

llm = init_chat_model(
    "llama-3.1-8b-instant",
    model_provider="groq"
)
llm


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001C2AC0EC770>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C2A9AEF500>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [24]:
llm.invoke("what is AI")

AIMessage(content='**Artificial Intelligence (AI)** is a field of computer science that focuses on creating intelligent machines that can think and act like humans. AI involves the development of algorithms and statistical models that enable computers to perform tasks that typically require human intelligence, such as:\n\n1. **Learning**: The ability to learn from data and improve performance over time.\n2. **Reasoning**: The ability to draw conclusions and make decisions based on data and rules.\n3. **Problem-solving**: The ability to identify and solve complex problems.\n4. **Perception**: The ability to interpret and understand data from sensors, such as images and speech.\n5. **Language understanding**: The ability to comprehend and generate human language.\n\nAI technologies are widely used in many areas, including:\n\n1. **Virtual assistants**: AI-powered virtual assistants, such as Siri, Alexa, and Google Assistant, can understand voice commands and perform tasks.\n2. **Image re

### Modern RAG chain

In [25]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


In [26]:
## Covert vector store to retriever
retriever = vectorstore.as_retriever(
    search_kwarg ={"k":3} # retrieve top 3 chunks
)
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001C2A6A565A0>, search_kwargs={})

In [27]:
## Implement a Prompt Template for the Retrieval Chain

system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("human","{input}")
])

In [28]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [29]:
### Create a document chain
document_chain = create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': Fa

### How this Chain Works

This chain:

1. Takes the retrieved documents.
2. "Stuffs" all retrieved documents into the prompt's `{context}` placeholder.
3. Sends the complete prompt (context + user question) to the LLM.
4. Returns the response generated by the LLM.

#### Flow

```text
Retrieved Documents
        ↓
Stuff Documents Chain
        ↓
Prompt Template ({context})
        ↓
LLM
        ↓
Generated Answer
```

In [30]:
### Create the final RAG chain 
rag_chain = create_retrieval_chain(retriever,document_chain)
rag_chain


RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001C2A6A565A0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you d

In [31]:
# run this rag 
response = rag_chain.invoke({"input":"What is Deep Learning"})

In [32]:
response ## it give full chain working output

{'input': 'What is Deep Learning',
 'context': [Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly\n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
  Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, an

In [33]:
response['answer'] ## LLM generated output

'Deep learning is a subset of machine learning based on artificial neural networks, inspired by the human brain and consisting of layers of interconnected nodes. It has revolutionized fields like computer vision, natural language processing, and speech recognition.'

In [34]:
# Function to query the modern RAG system

def query_rag_modern(question):

    print(f"Question: {question}")
    print("-" * 50)

    # Using create_retrieval_chain approach
    result = rag_chain.invoke({"input": question})

    print(f"Answer: {result['answer']}")

    print("\nRetrieved Context:")
    for i, doc in enumerate(result['context']):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

    return result


# Test queries
test_questions = [
    "What are the three types of machine learning?",
    "What is deep learning and how does it relate to neural networks?",
    "What are CNNs best used for?"
]


for question in test_questions:
    result = query_rag_modern(question)
    print("\n" + "=" * 80 + "\n")

Question: What are the three types of machine learning?
--------------------------------------------------
Answer: The three main types of machine learning are: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning learns through interaction.

Retrieved Context:

--- Source 1 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are ...

--- Source 2 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are ...

--- Source 3 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to le

### Create RAG Chain ALternative - Using LCEL (LangChain Expression Language) means without in-built lib

```text
# RAG Chain Architecture

┌─────────────┐
│ User Query  │
└──────┬──────┘
       │
       ▼
┌─────────────┐
│ Embeddings  │
└──────┬──────┘
       │
       │ Similarity Search
       ▼
┌─────────────────┐
│ Vector Store    │
│   (ChromaDB)    │
└──────┬──────────┘
       │
       │ Retrieve Relevant Chunks
       ▼
┌─────────────┐
│   Context   │
└──────┬──────┘
       │
       ▼
┌─────────────┐
│     LLM     │
└──────┬──────┘
       │
       ▼
┌─────────────┐
│   Answer    │
└─────────────┘
```

In [35]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
# RunnablePassthrough:
# Passes input unchanged to the next step in the chain

# RunnableParallel:
# Runs multiple chain components simultaneously
# and combines their outputs into a dictionary

In [36]:
# create a custom prompt 
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question.
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

context:
{context}

Question: {question}
Answer:""")

custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question.\nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\ncontext:\n{context}\n\nQuestion: {question}\nAnswer:"), additional_kwargs={})])

In [37]:
# above i coverted vector search to retriever so i directly use 
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001C2A6A565A0>, search_kwargs={})

In [38]:
## Format the output documents for the prompt <-- This code is for stuffing the chunks in one document 
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [39]:
## Build the chain using LCEL.

rag_chain_lcel = (
    {
        "context":retriever | format_docs,
        "question":RunnablePassthrough()
    }
    | custom_prompt        
    | llm                  
    |StrOutputParser()       
    
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001C2A6A565A0>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question.\nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\ncontext:\n{context}\n\nQuestion: {question}\nAnswer:"), additional_kwargs={})])
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'vid

In [40]:
# now run this rag chain
response = rag_chain_lcel.invoke("What is Deep Learning")
response

'Based on the provided context, Deep Learning is a subset of machine learning based on artificial neural networks. \n\nSpecific details from the context to support this answer include:\n- The first paragraph states: "Deep learning is a subset of machine learning based on artificial neural networks."\n- The subsequent paragraphs provide additional information about artificial neural networks, but do not add any new information about the definition of Deep Learning.'

In [41]:
docs = retriever.invoke("What is Deep Learning")

for doc in docs:
    print(doc.page_content)
    print("-" * 50)

Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of interconnected
    nodes. Deep learning has revolutionized fields like computer vision, natural language
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers
--------------------------------------------------
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of interconnected
    nodes. Deep learning has revolutionized fields like computer vision, natural language
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly
    effective for image processing, while Recurrent Neural Netw

In [42]:
docs = retriever.invoke("What is Deep Learning")
docs

[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly\n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Netwo

In [43]:
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)

    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")

    docs = retriever.invoke(question)

    print("\nSource Documents:")

    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [44]:
query_rag_lcel("What is Deep Learning?")

Question: What is Deep Learning?
--------------------------------------------------
Answer: Deep learning is a subset of machine learning based on artificial neural networks. 

Supporting details from the context:
- The context states "Deep learning is a subset of machine learning based on artificial neural networks."
- This information is repeated in multiple instances within the provided context, solidifying the fact that Deep Learning is a subset of machine learning based on artificial neural networks.

Source Documents:

--- Source 1 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of in...

--- Source 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of in...

--- Source 3 ---
Deep Lea

In [45]:
docs = retriever.invoke(question)

In [46]:
docs

[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly\n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Netwo

### Add New Documents To Existing Vector Store

In [47]:
vectorstore

In [48]:
# Add new documents to the existing vector store

new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make
decisions by interacting with an environment. The agent receives rewards or penalties
based on its actions and learns to maximize cumulative reward over time. Key concepts
in RL include: states, actions, rewards, policies, and value functions. Popular RL
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),
robotics, and autonomous systems.
"""

In [49]:
new_document

'\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make\ndecisions by interacting with an environment. The agent receives rewards or penalties\nbased on its actions and learns to maximize cumulative reward over time. Key concepts\nin RL include: states, actions, rewards, policies, and value functions. Popular RL\nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and\nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),\nrobotics, and autonomous systems.\n'

In [50]:
new_doc = Document(
    page_content=new_document,
    metadata={"source":"manual_addition","topic":"reinforcement_Learning"}
)

In [51]:
new_doc

Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_Learning'}, page_content='\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make\ndecisions by interacting with an environment. The agent receives rewards or penalties\nbased on its actions and learns to maximize cumulative reward over time. Key concepts\nin RL include: states, actions, rewards, policies, and value functions. Popular RL\nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and\nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),\nrobotics, and autonomous systems.\n')

In [52]:
new_chunks =text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_Learning'}, page_content='Reinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make\ndecisions by interacting with an environment. The agent receives rewards or penalties\nbased on its actions and learns to maximize cumulative reward over time. Key concepts\nin RL include: states, actions, rewards, policies, and value functions. Popular RL\nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and\nActor-Critic methods. RL has been'),
 Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_Learning'}, page_content='methods, and\nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),\nrobotics, and autonomous systems.')]

In [53]:
### how to add new document to vector store 
vectorstore.add_documents(new_chunks)
# use this .add_documents() function to add in existing vector search 

['3ecd2476-89c8-4930-aaf2-d2cefb6902ca',
 'a84e4ea0-5b1d-4b9f-9ebb-81d62d013522']

In [54]:
print(f"Added {len(new_chunks)} new chunks to the vector store")
print(f"Total vector now :{vectorstore._collection.count()}")

Added 2 new chunks to the vector store
Total vector now :36


In [55]:
# query with the updated vector
new_question="what are the key concept in reinforcement learning"
result =query_rag_lcel(new_question)

Question: what are the key concept in reinforcement learning
--------------------------------------------------
Answer: The key concepts in reinforcement learning include: 

states, actions, rewards, policies, and value functions. 
These are mentioned in the provided context as "Key concepts in RL include: states, actions, rewards, policies, and value functions."

Source Documents:

--- Source 1 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make
decisions by interacting with an environment. The agent receives rewards or pe...

--- Source 2 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make
decisions by interacting with an environment. The agent receives rewards or pe...

--- Source 3 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make
decisions by interacting with a

### Coversational Memory --[Add previous conversation context]

## Advanced RAG --> Coversational Memory

### Key Components

#### 1. `create_history_aware_retriever`
- Makes the retriever understand the conversation context.
- Uses previous chat messages to better understand the current user question.
- Helps retrieve more relevant documents from the Vector Database.

**Example:**

User: *Who is Elon Musk?*  
User: *Which company does he own?*

Without chat history → Retriever may not know who **"he"** refers to.

With `create_history_aware_retriever` → Retriever understands that **"he" = Elon Musk** and retrieves relevant documents.

---

#### 2. `MessagesPlaceholder`
- A placeholder used to insert chat history into the prompt.
- Allows the LLM to see previous conversation messages.

**Example:**

```python
MessagesPlaceholder("chat_history")
```

This tells LangChain:

> "Insert all previous conversation messages here."

---

#### 3. `HumanMessage`
- Represents a message sent by the user.
- Used to store user questions in chat history.

**Example:**

```python
HumanMessage(content="What is Machine Learning?")
```

---

#### 4. `AIMessage`
- Represents a message sent by the AI.
- Used to store AI responses in chat history.

**Example:**

```python
AIMessage(content="Machine Learning is a subset of AI.")
```

---

### Simple Flow

User Question
↓
Chat History (`HumanMessage` + `AIMessage`)
↓
`MessagesPlaceholder`
↓
`create_history_aware_retriever`
↓
Retriever understands context
↓
Vector DB Search
↓
Relevant Documents Retrieved
↓
LLM Generates Better Answer

In [56]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage

In [57]:
# create a prompt that includes the chat history
contextualized_q_system_prompt = """Give a chat history and the latest user question
which might reference context in the next chat history, formulate a standalone question
which can be understood without the chat history. Do NOT answer the question,
just reformulate it if needed and otherwise return it as is.""" 

contextualized_q__prompt = ChatPromptTemplate.from_messages([
    ("system", contextualized_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}")
])



In [58]:
## create history aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualized_q__prompt
)

In [59]:
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001C2A6A565A0>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag

In [60]:
# create a new document chain with history aware retriever
qa_system_prompt = """ You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

context:{context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}")
])

question_answering_chain = create_stuff_documents_chain(llm,qa_prompt)

# create conversational RAG chain
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answering_chain
)

print("Conversational RAG Chain created successfully!")

Conversational RAG Chain created successfully!


In [61]:
# First question

chat_history = []
result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is Machine Learning?"

})

print("Q: What is Machine Learning?")
print(f"A: {result1['answer']}")

Q: What is Machine Learning?
A: Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It allows systems to analyze data, identify patterns, and make predictions or decisions based on that data. This process enables the system to learn and adapt over time.


In [64]:
chat_history.extend([
    HumanMessage(content="What is Machine Learning?"),
    AIMessage(content=result1['answer'])
])

In [65]:
# followup  question

result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input":"What are its main types?"

})

result2

{'chat_history': [HumanMessage(content='What is Machine Learning?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It allows systems to analyze data, identify patterns, and make predictions or decisions based on that data. This process enables the system to learn and adapt over time.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'input': 'What are its main types?',
 'context': [Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning us

In [66]:
result2['answer']

'There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data, unsupervised learning finds patterns in unlabeled data, and reinforcement learning learns through interaction.'

# History Aware Retriever Flow

```text
User Question
      ↓
Check Chat History Available?
      ↓

 ┌────────────────────┬────────────────────┐
 │                    │                    │
No History         History Available
 │                    │
 │                    │
Direct Search      LLM Rewrites Question
 │                    │
 │                    │
"What is AI?"      "What are its uses?"
                       ↓
                "What are the uses of AI?"
                       ↓

 └───────────────↓────────────────┘

          Retriever Search
         (Chroma Vector DB)

                  ↓

        Relevant Documents
```

## How It Works

### Case 1: No Chat History

**User Question:**

```text
What is Deep Learning?
```

**Flow:**

```text
Question
   ↓
Retriever
   ↓
Search Chroma DB
   ↓
Relevant Documents
```

---

### Case 2: Chat History Available

**Conversation:**

```text
User: What is Deep Learning?

AI: Deep Learning is a subset of Machine Learning.

User: What are its applications?
```

**Flow:**

```text
"What are its applications?"
            ↓
LLM checks chat history
            ↓
"What are the applications of Deep Learning?"
            ↓
Retriever Search
            ↓
Relevant Documents
```

---

## Why Use `create_history_aware_retriever()`?

Without chat history:

```text
What are its applications?
```

Retriever does not know what **"its"** refers to.

With chat history:

```text
What are the applications of Deep Learning?
```

The query becomes clear and search results become more accurate.

---

## One-Line Summary

```text
No Chat History
      ↓
Direct Search

Chat History Available
      ↓
Rewrite Question using LLM
      ↓
Search Vector Database
      ↓
Retrieve Relevant Documents
```